# SimOC — End-to-End Demo

**Agent-based simulation of object-centric processes, discovered from OCEL 2.0 event logs.**

Four cells, four visuals. Each cell is one pipeline stage.

## 1. The Input — Real OCEL 2.0 Event Log

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import numpy as np
from simoc.ingestion import load_ocel

data = load_ocel("data/sample_order_process.json")

# --- Timeline plot: each object's lifecycle as a horizontal line ---
type_colors = {"order": "#e74c3c", "item": "#3498db", "delivery": "#2ecc71"}
type_y = {"order": 3, "item": 2, "delivery": 1}
oid_to_type = data.objects.set_index("object_id")["object_type"].to_dict()

fig, ax = plt.subplots(figsize=(12, 4))
t0 = data.events["timestamp"].min()

for oid, lc in data.lifecycles.items():
    otype = oid_to_type[oid]
    color = type_colors[otype]
    y_base = type_y[otype]
    y = y_base + hash(oid) % 100 / 250  # slight vertical jitter

    hours = [(ts - t0).total_seconds() / 3600 for _, _, ts in lc]
    ax.plot(hours, [y] * len(hours), color=color, alpha=0.6, linewidth=1.5)
    ax.scatter(hours, [y] * len(hours), color=color, s=20, zorder=5)

    # Label first point
    ax.text(hours[0] - 0.3, y + 0.08, oid.split("_")[0][0] + oid[-1],
            fontsize=6, color=color, ha="right")

ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["delivery", "item", "order"], fontsize=11)
ax.set_xlabel("Time (hours)", fontsize=11)
ax.set_title("Real Event Log — Object Lifecycles Over Time", fontsize=13, fontweight="bold")
ax.set_xlim(-1, 11)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 2. What SimOC Discovers — The Process Blueprint

In [ ]:
from simoc.discovery.interaction_graph import discover_interaction_graph
from simoc.discovery.cardinality import compute_spawning_profiles
from simoc.discovery.behavioral import discover_behavioral
from simoc.discovery.patterns import discover_patterns
from simoc.discovery.data_structures import DiscoveryResult

bd, oig, tc = discover_interaction_graph(data)
sp = compute_spawning_profiles(data, bd, tc)
bp = discover_behavioral(data, bd, tc)
ip = discover_patterns(data, bd, tc)
dr = DiscoveryResult(bd, oig, tc, sp)

# --- Dependency diagram ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis("off")

# Boxes for each type
boxes = {"order": (2, 4), "item": (5, 4), "delivery": (5, 1.5)}
for t, (x, y) in boxes.items():
    role = tc.classification[t]
    color = "#e74c3c" if role == "root" else "#3498db"
    rect = plt.Rectangle((x-0.8, y-0.4), 1.6, 0.8, facecolor=color, alpha=0.2,
                          edgecolor=color, linewidth=2, zorder=3)
    ax.add_patch(rect)
    ax.text(x, y+0.05, t.upper(), ha="center", va="center", fontsize=12,
            fontweight="bold", color=color, zorder=4)
    ax.text(x, y-0.25, role, ha="center", va="center", fontsize=8, color="gray", zorder=4)

# Spawning arrows
for (pt, ct), profile in sp.items():
    x1, y1 = boxes[pt]; x2, y2 = boxes[ct]
    ax.annotate("", xy=(x2-0.8, y2), xytext=(x1+0.8, y1),
                arrowprops=dict(arrowstyle="->", color="#333", lw=2))
    mx, my = (x1+x2)/2, (y1+y2)/2 + 0.2
    ax.text(mx, my, f"spawns\n{np.mean(profile.raw_counts):.1f}/parent",
            ha="center", fontsize=8, color="#333",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="#ccc"))

# Sync annotation
for (act, stype), rule in ip.synchronization_rules.items():
    if stype == "item":
        ax.text(6.5, 4.8, f"SYNC at {act}\ncondition: {rule.condition}\ndelay: {np.mean(rule.raw_sync_delays)/60:.0f} min",
                fontsize=8, color="#8e44ad", ha="left",
                bbox=dict(boxstyle="round", facecolor="#f5eef8", edgecolor="#8e44ad"))

# Release annotation
for (t1, t2), rule in ip.release_rules.items():
    ax.text(6.5, 3.2, f"RELEASE ({t1}, {t2})\nat {rule.release_activity}",
            fontsize=8, color="#e67e22",
            bbox=dict(boxstyle="round", facecolor="#fef5e7", edgecolor="#e67e22"))

# Binding annotation
if ip.binding_policies:
    ax.text(6.5, 1.5, f"BINDING: {len(ip.binding_policies)} policies",
            fontsize=8, color="#27ae60",
            bbox=dict(boxstyle="round", facecolor="#eafaf1", edgecolor="#27ae60"))

ax.set_title("Discovered Process Blueprint — Types, Spawning, Interactions",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Simulation — Synthetic Log From Discovered Model

In [ ]:
from simoc.simulation.runner import SimulationRunner
from simoc.evaluation._helpers import sim_result_to_oceldata
import pandas as pd

duration = (data.events["timestamp"].max() - data.events["timestamp"].min()).total_seconds()
runner = SimulationRunner(dr, bp, ip)
result = runner.run(duration=duration, seed=42)
syn = sim_result_to_oceldata(result)

syn_oid_to_type = syn.objects.set_index("object_id")["object_type"].to_dict()
syn_t0 = syn.events["timestamp"].min()

# --- Side-by-side timeline: Real vs Synthetic ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, d, d_oid_type, d_t0, title in [
    (ax1, data, oid_to_type, t0, f"Real Log ({len(data.events)} events)"),
    (ax2, syn, syn_oid_to_type, syn_t0, f"SimOC Synthetic ({len(syn.events)} events)"),
]:
    for oid, lc in d.lifecycles.items():
        otype = d_oid_type.get(oid, "")
        if otype not in type_colors:
            continue
        color = type_colors[otype]
        y = type_y.get(otype, 0) + hash(oid) % 100 / 250
        hours = [(ts - d_t0).total_seconds() / 3600 for _, _, ts in lc]
        ax.plot(hours, [y] * len(hours), color=color, alpha=0.4, linewidth=1)
        ax.scatter(hours, [y] * len(hours), color=color, s=10, zorder=5)

    ax.set_yticks([1, 2, 3])
    ax.set_yticklabels(["delivery", "item", "order"], fontsize=10)
    ax.set_xlabel("Time (hours)", fontsize=10)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)

plt.suptitle("Real vs Synthetic — Do the patterns match?", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 4. How Close? — SimOC vs Baselines

In [ ]:
from simoc.evaluation.baselines import build_flat_simod_runner, build_independent_runner
from simoc.evaluation.metrics import compute_all_metrics

# Run SimOC + 2 baselines
flat = build_flat_simod_runner(data, dr, bp, case_type="order")
indep = build_independent_runner(data, dr, bp)

flat_result = flat.run(duration=duration, seed=42)
indep_result = indep.run(duration=duration, seed=42)

m_simoc = compute_all_metrics(data, syn)
m_flat = compute_all_metrics(data, sim_result_to_oceldata(flat_result))
m_indep = compute_all_metrics(data, sim_result_to_oceldata(indep_result))

# --- Radar chart ---
# Select 5 key metrics, normalize so higher = better for all
labels = ["Activity\nEMD", "Duration\nKS pass", "Arrival\nerror", "OC-DFG\nsimilarity", "O2O\nfidelity"]
keys = ["activity_frequency_emd", "duration_ks_pass_rate", "arrival_rate_error",
        "oc_dfg_cosine_similarity", "o2o_fidelity"]

def to_radar(m):
    """Convert metrics to [0,1] where 1 = best."""
    vals = []
    for k in keys:
        v = m[k]
        if k in ("activity_frequency_emd", "arrival_rate_error"):
            vals.append(max(0, 1.0 - v))  # invert: lower EMD/error = better
        else:
            vals.append(v)  # higher = better
    return vals

vals_simoc = to_radar(m_simoc)
vals_flat = to_radar(m_flat)
vals_indep = to_radar(m_indep)

angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]  # close polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for vals, name, color, ls in [
    (vals_simoc, "SimOC", "#e74c3c", "-"),
    (vals_flat, "Flat Simod", "#95a5a6", "--"),
    (vals_indep, "Independent", "#3498db", "-."),
]:
    v = vals + vals[:1]
    ax.plot(angles, v, color=color, linewidth=2, linestyle=ls, label=name)
    ax.fill(angles, v, color=color, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=7, color="gray")
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.set_title("Simulation Fidelity — Higher is Better", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()